# Classification metrics from scores & probabilities

_Metrics & Evaluation — measuring models, data & statistics_

**Your model outputs a number per example. These metrics grade how good that number is — before you pick any yes/no cutoff.**

A classifier rarely says a flat "yes" or "no". Under the hood it produces a score per example — a number where higher means "more likely to be a positive". A probability is a score that has been squeezed into the 0–1 range so it can be read as "an X% chance".

       There are really two different questions you can ask about such scores, and they need different metrics:

---

This notebook is a practice scaffold for the **Classification metrics from scores & probabilities** lesson. We assemble the reference code one step at a time — run each cell, read the short note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. Each row is one example; the columns are its features, plus a class label.

In [ ]:
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer(as_frame=True)
print("rows x columns:", data.frame.shape)
print("feature columns:", list(data.data.columns))
print("target classes :", list(data.target_names))
data.frame.head()

## Reference implementation — scikit-learn

### Step 1 — Train a model and read out its probabilities

We split the data, **standardize** the features (logistic regression fits more stably on scaled inputs), and fit the classifier. The crucial call is `predict_proba(...)[:, 1]`, which gives the model's estimated probability of the **positive** class (here `1` = benign) for each test example. Every metric below grades that vector of probabilities `p`.

In [ ]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X, y = load_breast_cancer(return_X_y=True)          # y: 1 = benign (positive), 0 = malignant
Xtr, Xte, ytr, yte = train_test_split(
    X, y, test_size=0.3, random_state=0, stratify=y)

clf = LogisticRegression(max_iter=5000)
clf.fit(StandardScaler().fit_transform(Xtr), ytr)   # scale for a stable fit
p = clf.predict_proba(StandardScaler().fit(Xtr).transform(Xte))[:, 1]

### Step 2 — Ranking metrics (threshold-free)

The first question is purely about **ordering**: does the model rank positives above negatives? ROC-AUC is the probability a random positive outscores a random negative; average precision (PR-AUC) summarizes the precision–recall curve; Gini just rescales AUC so random = 0; and the KS statistic is the largest gap between the true- and false-positive rates. None of these needs you to pick a cutoff.

In [ ]:
from sklearn.metrics import (roc_curve, roc_auc_score,
                             precision_recall_curve, average_precision_score)

# --- ranking metrics (threshold-free) ---
auc  = roc_auc_score(yte, p)                        # P(random positive outranks random negative)
ap   = average_precision_score(yte, p)              # PR-AUC; baseline = positive rate
gini = 2 * auc - 1                                  # rescale: random = 0, perfect = 1
fpr, tpr, roc_t = roc_curve(yte, p)
ks   = np.max(tpr - fpr)                            # Kolmogorov-Smirnov statistic
prec, rec, pr_t = precision_recall_curve(yte, p)

print(f"ROC-AUC {auc:.4f}  PR-AUC/AP {ap:.4f}  Gini {gini:.4f}  KS {ks:.4f}")

### Step 3 — Probability quality and picking a threshold

The second question is whether the probabilities are **honest**. Log loss (cross-entropy) punishes confident mistakes; the Brier score is just the mean squared error of the probabilities. Finally, if you *do* need a yes/no cutoff, Youden's J — the threshold that maximizes `TPR - FPR`, the same KS point on the ROC curve — is a common default.

In [ ]:
from sklearn.metrics import log_loss, brier_score_loss

# --- probability-quality metrics ---
ll   = log_loss(yte, p)                             # cross-entropy; punishes confident mistakes
brier = brier_score_loss(yte, p)                    # mean squared error of the probabilities

print(f"log loss {ll:.4f}  Brier {brier:.4f}")

# --- choose a threshold: Youden's J (the KS point on the ROC curve) ---
best = roc_t[np.argmax(tpr - fpr)]
print(f"threshold maximizing TPR - FPR: {best:.3f}")

## Visualize the data & results

_On a real classifier (logistic regression on load_breast_cancer), how do the ROC and Precision–Recall curves look, and what do their areas (AUCs) say?_

### Step 1 — Refit the model and get test probabilities

We rebuild the same standardized logistic-regression model so the visualization stands on its own. Note we fit the scaler once on the training data and reuse it on the test set, then pull the positive-class probabilities `p` to feed the curves.

In [ ]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X, y = load_breast_cancer(return_X_y=True)
Xtr, Xte, ytr, yte = train_test_split(
    X, y, test_size=0.3, random_state=0, stratify=y)
sc = StandardScaler().fit(Xtr)
clf = LogisticRegression(max_iter=5000).fit(sc.transform(Xtr), ytr)
p = clf.predict_proba(sc.transform(Xte))[:, 1]

### Step 2 — Print the headline ranking numbers

Before plotting, look at the summary scores. The positive rate (0.626) is the baseline an average-precision number must beat. Both ROC-AUC and average precision are near 1.0 here — this is an easy dataset, so the curves below will hug the ideal corner.

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score

print("positive rate:", round(yte.mean(), 3))           # 0.626
print("ROC-AUC:", round(roc_auc_score(yte, p), 4))      # 0.9956
print("Avg Precision:", round(average_precision_score(yte, p), 4))  # 0.9974

### Step 3 — Compute the ROC and PR curves

`roc_curve` and `precision_recall_curve` return many points — one per distinct threshold. The `thin` helper evenly subsamples about a dozen of them so the printed list is readable. Each ROC point is an (FPR, TPR) pair; each PR point is a (recall, precision) pair.

In [ ]:
from sklearn.metrics import roc_curve, precision_recall_curve

fpr, tpr, _ = roc_curve(yte, p)
prec, rec, _ = precision_recall_curve(yte, p)

def thin(a, b, n=12):                                   # subsample for plotting
    idx = np.linspace(0, len(a) - 1, n).astype(int)
    return [[round(float(a[i]), 3), round(float(b[i]), 3)] for i in idx]

print("ROC points:", thin(fpr, tpr))
print("PR  points:", thin(rec, prec))

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** A fraud model on data that is 1% fraud reports ROC-AUC = 0.96 but Average Precision (PR-AUC) = 0.22. Is this a good model, and which number do you trust?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall what each metric is denominated in. — _ROC-AUC uses FPR = FP/(FP+TN); with 99% negatives, TN is huge, so false alarms barely register and ROC-AUC stays high._
- Compare AP to the right baseline. — _A no-skill model's AP equals the positive rate, here 0.01. So AP = 0.22 is 22× better than random — real signal, but far from the "0.96" ROC implies._
- Decide what matters operationally. — _If you act on flagged cases, precision (PR-AUC's currency) is what you live with. The 0.96 is hiding a wall of false positives._

**Answer:** The model has genuine signal but is mediocre at the precision you'll actually experience. Under 1% imbalance, trust the PR-AUC (0.22 vs a 0.01 baseline); the ROC-AUC of 0.96 is the classic "looks great under imbalance" mirage.

</details>

**Problem 2.** Two models have the same ROC-AUC = 0.90. Model A has log loss 0.35; Model B has log loss 0.80. You need to multiply each predicted probability by a dollar amount. Which model do you ship?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Separate ranking from probability quality. — _Equal ROC-AUC means equal ranking ability — both order positives above negatives equally well._
- Read the log loss as a calibration signal. — _Log loss grades the actual probabilities. B's higher loss means its probabilities are less honest (likely overconfident)._
- Match the metric to the use case. — _Multiplying a probability by dollars needs the number to mean what it says, so probability quality is decisive here._

**Answer:** Ship Model A. Identical ROC-AUC says they rank equally, but A's much lower log loss means its probabilities are better calibrated — and since you multiply probabilities by money, probability quality (log loss / Brier), not ranking, is what counts. See [met-calibration].

</details>

**Problem 3.** Your model gives ROC-AUC = 0.88 but a stakeholder asks "what's the precision and recall of the system we're actually deploying?" Why can't you answer from the AUC, and what do you do?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- State what AUC summarizes. — _ROC-AUC averages over every possible threshold; it deliberately does not commit to one._
- Identify the missing decision. — _Precision and recall are properties of a single chosen threshold, which the AUC never picked._
- Choose a threshold deliberately. — _Use a cost-based cutoff, Youden's J (the KS point), or fix a required precision/recall — then read off the confusion-matrix numbers there._

**Answer:** AUC is threshold-free, so it says nothing about any single operating point. To answer, you must pick a threshold (by cost, by Youden's J / KS, or by a required precision) and then report precision and recall computed at that threshold.

</details>